In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel

from dotenv import load_dotenv
import os
load_dotenv()


if os.environ['GEMINI_API_KEY']:
    print('gemini key set')
else:
    print("not set")

gemini key set


# **Chain With Parallel Chains**

In [46]:

# Task 1, [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie summarizer"),
    ("user", "Please summarize the movie in brief: {input}")
])


In [47]:
# Task 2, LLM

llm_task = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [48]:
# Task 3, [String Parser]
str_parser = StrOutputParser()

In [49]:
# Task 4, [Custom Runnable]

def dictionary_maker(text:str) -> dict:
    return {"text":text}


dictionary_maker_runnable = RunnableLambda(dictionary_maker)

## **Parallel Chain 1**

In [50]:
# Task -1 [Prompt chain 1]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You're a linkedin post generator"),
    ("user","Create a post for the following text for LinkedIn: {text}")
])

# task -2  chain 1

llm_chain_1 = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# Task 3 string parser chain 1, [String Parser]
str_parser_chain_1 = StrOutputParser()


chain_linkedin = linkedin_prompt | llm_chain_1 | str_parser_chain_1

## **Parallel Chain 2**

In [51]:
def insta_chain(text:dict):
    # Task -1 [Prompt chain 1]

    # text = text['text']

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","You're an instagram post generator"),
        ("user","Create a post for the following text for instagram: {text}")
    ])

    # task -2  chain 1

    llm_chain_2 = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )


    # Task 3 string parser chain 1, [String Parser]
    str_parser_chain_2 = StrOutputParser()

    chain_insta = insta_prompt | llm_chain_2 | str_parser_chain_2

    # result = chain_insta.invoke(text)

    return chain_insta # result


insta_chain_runnable = RunnableLambda(insta_chain)

## **Final Orchestration**

In [52]:
final_chain = (prompt_template |
              llm_task | 
              str_parser | 
              dictionary_maker_runnable |
              RunnableParallel(branches = {"linkedin":chain_linkedin, "instagram":insta_chain_runnable}))


# final_response = final_chain

## **Chain As a Runnable**

In [54]:
# Task 1, [Beautify Function]
def beautify(final_response:dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['instagram']

    return {'linkedin':linkedin_response,'instagram':insta_response}

beautify_runnable = RunnableLambda(beautify)

chain_final = final_chain | beautify_runnable

chain_final.invoke("Transformers age of extinction")

{'linkedin': 'Here are three options for your LinkedIn post, depending on the "vibe" you want to go for:\n\n### Option 1: The "Leadership & Resilience" Angle (Best for engagement)\n**Headline: What a struggling inventor can teach us about leadership.**\n\nIn *Transformers: Age of Extinction*, Cade Yeager isn’t a soldier or a government official. He’s an inventor facing a world that has turned its back on his allies. \n\nWhen he discovers a dormant Optimus Prime, he’s faced with a choice: stay safe in the shadows or step into a global conflict to protect those who can’t protect themselves.\n\nThis story arc is a masterclass in:\n✅ **Resourcefulness:** Turning a "piece of junk" into a catalyst for change.\n✅ **Loyalty:** Standing by your team even when the public narrative shifts against them.\n✅ **Adaptability:** When the old ways don\'t work, you don\'t just quit—you recruit the Dinobots.\n\nSometimes, the biggest breakthroughs come when you’re at your lowest point. Are you looking for